# Slice 2 — Kaggle verification

Use this notebook with the updated private handoff input. It targets the reported
Python 3.13.16 / PyTorch 2.11.0+cu130 / CUDA 13.0 / two Tesla T4 runtime.

**First session:** keep MODE='verify' below, enable Internet and GPU T4 x2, then Run All.
Setup installs only the bundled project wheel and a missing bundled datasketch wheel.
Existing scientific packages are checked and preserved. CPU/tiny-GPU checks precede
the pinned DistilBERT check and independent synthetic workers. All messages are invented.

**Second session:** download the first results ZIP, end the session, attach that ZIP
as a private input in a new session with the same handoff, change MODE='resume', Run All.
Download and return both sessions' result ZIPs. No HumAID data is used here.


In [ ]:
MODE = "verify"  # First session: "verify". Second session: "resume".
RUN_PRETRAINED = True
RUN_PAIR = True
RESULTS_DIR = None  # Auto-discover the first-session private input.
GPU_HOURS_BEFORE = None  # Enter the allowance shown in your Kaggle account.
GPU_HOURS_AFTER = None   # Update before rerunning the final export cell.
SESSION_LIMIT = None    # Enter the session limit shown, if available.


In [ ]:
"""Standard-library bootstrap embedded verbatim into the Kaggle notebook."""
import hashlib
import importlib.metadata as package_metadata
import json
import os
from pathlib import Path
import platform
import runpy
import shutil
import stat
import subprocess
import sys
import zipfile

HANDOFF_KIND = "slice2-kaggle-v2"

def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def extract_zip(path, target):
    """Regular files only; never write outside a fresh staging directory."""
    target = Path(target).resolve()
    target.mkdir(parents=True, exist_ok=False)
    try:
        with zipfile.ZipFile(path) as archive:
            names = archive.namelist()
            if len(names) != len(set(names)):
                raise ValueError("duplicate archive paths")
            for item in archive.infolist():
                destination = (target / item.filename).resolve()
                kind = stat.S_IFMT(item.external_attr >> 16)
                if (not destination.is_relative_to(target) or
                    item.filename.startswith("/") or
                    kind not in (0, stat.S_IFREG, stat.S_IFDIR)):
                    raise ValueError("archive contains unsupported paths or links")
            archive.extractall(target)
    except BaseException:
        shutil.rmtree(target)
        raise
    return target

def locate_handoff(input_root, working_root):
    input_root, working_root = Path(input_root), Path(working_root)
    override = os.environ.get("TRIAGE_HANDOFF_DIR")
    if override:
        return Path(override).resolve()
    candidates = []
    for path in input_root.rglob("handoff-manifest.json"):
        manifest = json.loads(path.read_text())
        if manifest.get("handoff_kind") == HANDOFF_KIND:
            candidates.append(path.parent)
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError("attach one current handoff or set TRIAGE_HANDOFF_DIR")
    archives = list(input_root.rglob("slice-2-kaggle-handoff.zip"))
    if len(archives) != 1:
        raise FileNotFoundError("Add the private handoff dataset to this notebook first")
    target = working_root / ("handoff-" + file_digest(archives[0])[:16])
    if not target.exists():
        extract_zip(archives[0], target)
    return target

def verify_handoff(bundle):
    bundle = Path(bundle).resolve()
    manifest = json.loads((bundle / "handoff-manifest.json").read_text())
    if manifest.get("handoff_kind") != HANDOFF_KIND:
        raise ValueError("use the updated Kaggle handoff")
    for name, expected in manifest["files"].items():
        raw_path = bundle / name
        path = raw_path.resolve()
        if not path.is_relative_to(bundle) or not path.is_file() or raw_path.is_symlink():
            raise ValueError("handoff member missing or outside bundle")
        if file_digest(path) != expected:
            raise ValueError(f"handoff digest mismatch: {name}")
    # Refuse unchecked executable extras in the source tree.
    actual = {p.relative_to(bundle).as_posix() for p in (bundle / "code").rglob("*") if p.is_file()}
    declared = {name for name in manifest["files"] if name.startswith("code/")}
    if actual != declared:
        raise ValueError("source tree contains unexpected or missing files")
    return manifest

def install_plan(bundle, *, python_version=None, installed=None):
    """Pure plan: drift fails before any package mutation."""
    bundle = Path(bundle).resolve()
    manifest = verify_handoff(bundle)
    profiles = runpy.run_path(str(bundle / "code/src/disaster_triage/profiles.py"))
    installed = profiles["installed_versions"]() if installed is None else dict(installed)
    errors = profiles["profile_errors"](
        "kaggle-py313", python_version or platform.python_version(), installed,
        allow_missing_datasketch=True,
    )
    if errors:
        raise RuntimeError("Runtime differs from the reported Kaggle profile:\n" + "\n".join(errors))
    commands = []
    if installed.get("datasketch") is None:
        commands.append([sys.executable, "-m", "pip", "install", "--no-deps",
                         str(bundle / manifest["datasketch_wheel"])])
    try:
        current_project = package_metadata.version("disaster-triage")
    except package_metadata.PackageNotFoundError:
        current_project = None
    project_matches = current_project == manifest["package_version"]
    if project_matches:
        distribution = package_metadata.distribution("disaster-triage")
        for source in (bundle / "code/src/disaster_triage").glob("*.py"):
            installed_file = Path(distribution.locate_file("disaster_triage/" + source.name))
            if not installed_file.is_file() or file_digest(installed_file) != file_digest(source):
                project_matches = False
                break
    if not project_matches:
        if "disaster_triage" in sys.modules:
            raise RuntimeError("Restart the kernel before replacing an already imported package")
        commands.append([sys.executable, "-m", "pip", "install", "--no-deps", "--force-reinstall",
                         str(bundle / manifest["project_wheel"])])
    return manifest, commands

def prepare_handoff(bundle, working_root, *, run_command=subprocess.check_call):
    manifest, commands = install_plan(bundle)
    for command in commands:
        run_command(command)
    # No already-imported scientific library is replaced. New package imports are safe.
    profiles = runpy.run_path(str(Path(bundle) / "code/src/disaster_triage/profiles.py"))
    errors = profiles["profile_errors"](
        "kaggle-py313", platform.python_version(), profiles["installed_versions"]())
    if errors:
        raise RuntimeError("\n".join(errors))
    code = Path(working_root) / ("code-" + manifest["source_sha256"][:16])
    if not code.exists():
        shutil.copytree(Path(bundle) / "code", code)
    return code, manifest

def dependency_closure():
    """Capture/check the project's installed dependency closure, without upgrades."""
    from packaging.requirements import Requirement
    from packaging.utils import canonicalize_name
    pending = ["disaster-triage"]
    records, errors = {}, []
    while pending:
        name = canonicalize_name(pending.pop())
        if name in records:
            continue
        try:
            distribution = package_metadata.distribution(name)
        except package_metadata.PackageNotFoundError:
            errors.append(f"missing dependency: {name}")
            continue
        requirements = []
        for value in distribution.requires or []:
            requirement = Requirement(value)
            if requirement.marker and not requirement.marker.evaluate({"extra": ""}):
                continue
            requirements.append(str(requirement))
            try:
                version = package_metadata.version(requirement.name)
                if not requirement.specifier.contains(version, prereleases=True):
                    errors.append(f"{requirement.name} {version} does not satisfy {requirement.specifier}")
            except package_metadata.PackageNotFoundError:
                errors.append(f"missing dependency: {requirement.name}")
            pending.append(requirement.name)
        records[name] = {"version": distribution.version, "requires": requirements}
    if errors:
        raise RuntimeError("\n".join(sorted(set(errors))))
    return records


def locate_results(input_root, working_root, override=None):
    if override:
        return Path(override).resolve()
    matches = []
    for path in Path(input_root).rglob("resume-index.json"):
        value = json.loads(path.read_text())
        if value.get("kind") == "slice2-verification-results" and value.get("phase") == "verify":
            matches.append(path.parent)
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise ValueError("attach one first-session result, or set RESULTS_DIR")
    archives = list(Path(input_root).rglob("slice-2-verification-results*.zip"))
    if len(archives) != 1:
        raise FileNotFoundError("Attach the downloaded first-session results as a private input")
    target = Path(working_root) / ("retrieved-" + file_digest(archives[0])[:16])
    if not target.exists():
        extract_zip(archives[0], target)
    return target

def export_results(working_root, *, phase, probes, evidence, source_sha256):
    """Export synthetic artifacts/accounting, excluding public model caches."""
    working_root = Path(working_root)
    artifacts = working_root / "artifacts"
    export = working_root / "return"
    export.mkdir(parents=True, exist_ok=True)
    index = {
        "kind": "slice2-verification-results", "schema_version": 1,
        "phase": phase, "source_sha256": source_sha256,
        "probes": [
            {"device": item["device"], "path": str(
                Path(item["verification_dir"]).relative_to(working_root))}
            for item in probes
        ],
    }
    (export / "resume-index.json").write_text(json.dumps(index, indent=2) + "\n")
    (export / "evidence.json").write_text(json.dumps(evidence, indent=2, allow_nan=False) + "\n")
    # Construct file list before opening output; never include caches or output itself.
    members = [(p, p.relative_to(working_root).as_posix())
               for p in artifacts.rglob("*") if p.is_file()]
    members += [(export / name, name) for name in ("resume-index.json", "evidence.json")]
    digests = {name: file_digest(path) for path, name in members}
    (export / "result-manifest.json").write_text(json.dumps(digests, indent=2) + "\n")
    target = export / "slice-2-verification-results.zip"
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path, name in members:
            archive.write(path, name)
        archive.write(export / "result-manifest.json", "result-manifest.json")
    return target

def verify_results(directory):
    directory = Path(directory).resolve()
    digests = json.loads((directory / "result-manifest.json").read_text())
    for name, expected in digests.items():
        path = (directory / name).resolve()
        if not path.is_relative_to(directory) or not path.is_file() or path.is_symlink():
            raise ValueError("retrieved result path invalid")
        if file_digest(path) != expected:
            raise ValueError(f"retrieved result digest mismatch: {name}")
    index = json.loads((directory / "resume-index.json").read_text())
    if index.get("kind") != "slice2-verification-results" or index.get("phase") != "verify":
        raise ValueError("attach the first-session verification results")
    return index


# Bootstrap is standard-library-only until verified package installation finishes.
if MODE not in ("verify", "resume"):
    raise ValueError("MODE must be verify or resume")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
WORK = Path("/kaggle/working/slice2")
BUNDLE = locate_handoff(Path("/kaggle/input"), WORK)
CODE, HANDOFF = prepare_handoff(BUNDLE, WORK)
print({"setup": "passed", "package": HANDOFF["package_version"],
       "profile": "kaggle-py313", "source_sha256": HANDOFF["source_sha256"]})


In [ ]:
import torch
from disaster_triage.config import RunConfig
from disaster_triage.runtime import doctor
from disaster_triage.provenance import save_json, source_snapshot
from disaster_triage.accounting import merge_accounting, budget_summary

ARTIFACTS = RunConfig().paths(WORK)["artifact_dir"]
merge_accounting(BUNDLE / "prior-accounting", ARTIFACTS)
runtime = doctor(RunConfig(device="cuda:0"), WORK)
assert runtime["python_supported"] and not runtime["profile_errors"], runtime["profile_errors"]
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Reported two-GPU allocation changed"
assert torch.version.cuda == "13.0", "CUDA build changed"
assert [torch.cuda.get_device_name(i) for i in range(2)] == ["Tesla T4", "Tesla T4"], "GPU allocation changed"
snapshot = source_snapshot(CODE, ARTIFACTS / "handoff-source.tar")
assert snapshot["code_sha256"] == HANDOFF["source_sha256"], "Extracted source changed"
closure = dependency_closure()
save_json(ARTIFACTS / "project-dependency-closure.json", closure)
pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
save_json(ARTIFACTS / "managed-pip-check.json",
          {"returncode": pip_check.returncode, "stdout": pip_check.stdout, "stderr": pip_check.stderr})
save_json(ARTIFACTS / "runtime-report.json", runtime)
print(runtime)
print({"project_dependency_closure": "passed",
       "managed_environment_pip_check_returncode": pip_check.returncode})
# The closure above must pass. The full managed image may report unrelated conflicts;
# retain its diagnostic without upgrading unrelated packages.


## Synthetic verification

The CPU and GPU probes train a tiny invented model for two optimizer windows and
verify recovery. The pretrained check uses two invented texts and downloads public
model assets. These checks provide compatibility evidence, not research results.


In [ ]:
from disaster_triage.smoke import run_offline, run_pretrained, resume_probe

probes = []
pretrained = None
paired = []
resumed = []
if MODE == "verify":
    probes.append(run_offline(RunConfig(device="cpu"), WORK, CODE))
    probes.append(run_offline(RunConfig(device="cuda:0"), WORK, CODE))
    print([{"device": p["device"], "recovery": p["recovery_equivalent"]} for p in probes])


In [ ]:
if MODE == "verify" and RUN_PRETRAINED:
    pretrained = run_pretrained(RunConfig(device="cuda:0"), WORK, CODE)
    print(pretrained)


In [ ]:
if MODE == "verify" and RUN_PAIR:
    commands = [
        [sys.executable, "-m", "disaster_triage", "smoke", "--root", str(WORK),
         "--source-root", str(CODE), "--device", f"cuda:{index}"]
        for index in (0, 1)
    ]
    workers = [subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
               for command in commands]
    failures = []
    for index, worker in enumerate(workers):
        output, error = worker.communicate()
        if worker.returncode:
            failures.append({"device": index, "stderr": error})
        else:
            paired.append(json.loads(output))
    save_json(ARTIFACTS / "paired-worker-diagnostics.json", {"failures": failures})
    if failures:
        raise RuntimeError("Independent synthetic worker failed; return its saved diagnostic")
    probes.extend(paired)
    print([{"device": p["device"], "recovery": p["recovery_equivalent"]} for p in paired])


## New-session retrieval proof

In the second session, attach the downloaded first-session results ZIP as a private
input. MODE='resume' skips new verification jobs and pretrained downloads, and instead
continues every retained CPU/GPU probe. This is separate from saving a checkpoint
inside a single still-running session.


In [ ]:
if MODE == "resume":
    retained = locate_results(Path("/kaggle/input"), WORK, RESULTS_DIR)
    index = verify_results(retained)
    merge_accounting(retained / "artifacts", ARTIFACTS)
    assert index["source_sha256"] == HANDOFF["source_sha256"], "Use the same handoff as session one"
    if not index["probes"]:
        raise ValueError("No retained verification probes found")
    for item in index["probes"]:
        path = (retained / item["path"]).resolve()
        assert path.is_relative_to(retained.resolve()), "Invalid retained probe path"
        resumed.append(resume_probe(RunConfig(device=item["device"]), WORK, CODE, path))
    print(resumed)


## Download the result ZIP

Set GPU_HOURS_AFTER above to the current allowance if available, then rerun this final
cell. Download the linked results ZIP before ending the session. Retain the first
session ZIP for the second-session input. Return both ZIPs for review.


In [ ]:
evidence = {
    "phase": MODE, "runtime": runtime, "pretrained": pretrained,
    "probes": probes, "paired_workers": paired, "retrieved_resumes": resumed,
    "accounting": budget_summary(ARTIFACTS),
    "quota_before": GPU_HOURS_BEFORE, "quota_after": GPU_HOURS_AFTER,
    "session_limit": SESSION_LIMIT,
    "source_sha256": HANDOFF["source_sha256"],
}
result_zip = export_results(WORK, phase=MODE, probes=probes, evidence=evidence,
                            source_sha256=HANDOFF["source_sha256"])
print({"phase": MODE, "results": str(result_zip),
       "active_job_hours": evidence["accounting"]["active_job_hours"],
       "retrieval_verified": MODE == "resume" and bool(resumed)})
from IPython.display import FileLink, display
display(FileLink(str(result_zip)))
